# 06 — Walking-network build (Stage 4a)

Downloads the Geofabrik Great Britain OSM PBF (one snapshot, ~2.1 GB), parses walkable highways inside the NE bounding box with pyosmium, computes geodesic edge lengths, and builds a `pandana.Network` ready for batch shortest-path queries.

Per DEC-008, we register the PBF in `data/manifest.csv` with its SHA256 so the routing graph is reproducible from the same snapshot.

## 0. Pre-flight

In [ ]:
import time
from datetime import datetime, timezone
from pathlib import Path

from schools_sunbeds import audit, config, network as nw

config.ensure_dirs()
audit.verify_manifest()

TODAY = datetime.now(timezone.utc).strftime("%Y%m%d")
OSM_DIR = config.DATA_RAW / "osm_extract" / TODAY
TODAY

## 1. Download the GB PBF (~2.1 GB; cached after first run)

In [ ]:
pbf_path = nw.fetch_geofabrik_gb_pbf(OSM_DIR)
audit.register_raw_file(
    pbf_path,
    source_url=nw.GEOFABRIK_GB_PBF_URL,
    licence="ODbL-1.0 (OpenStreetMap)",
    notes=f"Geofabrik GB PBF snapshot for {TODAY}; bbox-filtered to NE on parse",
)
print(f"PBF: {pbf_path.stat().st_size:,} bytes")

## 2. Parse the walking network for the NE bbox

The bbox is `config.REGION_BBOX_WGS84` with a small margin already baked in. We keep the network slightly wider than the LAD union so routes near the boundary are not artificially truncated.

In [ ]:
import geopandas as gpd

t0 = time.time()
data = nw.parse_walking_network(pbf_path, bbox_wgs84=config.REGION_BBOX_WGS84)
print(f"Parsed in {time.time()-t0:.1f}s; bbox-only: {len(data.nodes):,} nodes, {len(data.edges):,} edges")

# The NE bbox is wider than the actual NE LAD union (it includes parts of
# Cumbria and Yorkshire) so we clip to the LAD polygon plus a 2 km buffer
# to keep boundary-crossing routes intact while shrinking the network
# to a tractable size for pandana CH on macOS.
lad_files = sorted(config.DATA_PROCESSED.glob("lad_ne_*.gpkg"))
if not lad_files:
    raise RuntimeError("No lad_ne_*.gpkg in data/processed/. Run notebook 02 first.")
lad = gpd.read_file(lad_files[-1])
data = nw.clip_walking_network_to_polygon(data, lad, buffer_m=2_000)

print(f"After polygon clip: {len(data.nodes):,} nodes, {len(data.edges):,} edges")
print(f"  Median edge length: {data.edges['length_m'].median():.1f} m")
print(f"  Total network length: {data.edges['length_m'].sum() / 1000:,.0f} km")

## 3. Persist parsed nodes/edges + build the pandana network

In [ ]:
net_dir = config.DATA_PROCESSED / f"walk_network_{TODAY}"
nodes_path, edges_path = nw.save_walking_network(data, net_dir)
for p, n in ((nodes_path, "walk_nodes parquet"), (edges_path, "walk_edges parquet")):
    audit.write_provenance_sidecar(
        audit.Provenance(
            output_path=p,
            inputs=(pbf_path,),
            notes=f"Stage 4a: {n}",
            random_seed=config.RANDOM_SEED,
        )
    )
print("Wrote:", nodes_path)
print("Wrote:", edges_path)

In [ ]:
t0 = time.time()
net = nw.build_pandana_network(data)
# pandana.Network() runs the contraction hierarchy as part of construction;
# an explicit .precompute() on a 1.5M+ node graph was OOM-prone on macOS,
# and set_pois() in catchment.py does the per-category indexing we need
# at query time, so we skip it here.
print(f"Built pandana.Network in {time.time()-t0:.1f}s")

## 4. Sanity check — single shortest-path query

Before we trust the network for thousands of OD pairs, verify one Newcastle → Gateshead short path looks sensible.

In [ ]:
# Newcastle Central Station -> Gateshead Civic Centre, ~1.6 km direct
import pandas as pd

origin = (-1.6178, 54.9685)  # Newcastle Central
destination = (-1.6005, 54.9628)  # Gateshead Civic Centre

from_node = net.get_node_ids([origin[0]], [origin[1]]).iloc[0]
to_node = net.get_node_ids([destination[0]], [destination[1]]).iloc[0]
dist = net.shortest_path_lengths([from_node], [to_node])[0]
print(f"Shortest walking distance Newcastle Central → Gateshead Civic Centre: {dist:.0f} m")
print("(Expect ~1500–2200 m; bridge crossing means it's not a straight line.)")

## Done

Outputs:
- `data/processed/walk_network_<date>/walk_nodes.parquet`
- `data/processed/walk_network_<date>/walk_edges.parquet`

Notebook 07 reloads these and runs the catchment assignment.